In [1]:
from enum import Enum

import numpy as np
import pygame

import gymnasium as gym
from gymnasium import spaces

import importlib

from DnB import DotsAndBoxes, get_render_desc, draw_board

pygame 2.6.1 (SDL 2.28.4, Python 3.10.19)
Hello from the pygame community. https://www.pygame.org/contribute.html


c:\Users\yhwlo\anaconda3\envs\haic\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


### Space 정의

In [2]:
# action space 정의
# action: (edge_type, row, col)
# edge_type: 0 (h_edge), 1 (v_edge)
# row: 0~5 (h_edge), 0~4 (v_edge) 
# col: 0~4 (h_edge), 0~5 (v_edge)
# 불가능한 행동은 action mask로 후에 처리

action_space = spaces.MultiDiscrete(nvec=[2,6,6], dtype=int)
print(action_space)
print(action_space.sample())

MultiDiscrete([2 6 6])
[1 0 4]


In [3]:
# Observation space 정의
# h_edges: 가로 선 6x5  (0: 없음, 1: 있음)
# v_edges: 세로 선 5x6  (0: 없음, 1: 있음)
# box_owner: 상자 소유자 5x5 (-1: 유저1, 0: 없음, 1: 유저2)]

observation_space = spaces.Dict(
        {
            "h_edges": spaces.Box(0, 1, shape=(6,5), dtype=bool),
            "v_edges": spaces.Box(0, 1, shape=(5,6), dtype=bool),
            "box_owner": spaces.Box(-1, 1, shape=(5,5), dtype=int)
        }
    )

In [4]:
print(observation_space['h_edges'].sample())
print(observation_space['v_edges'].sample())
print(observation_space['box_owner'].sample())

[[False  True False  True  True]
 [ True  True  True  True False]
 [False  True False False  True]
 [False False  True False False]
 [False  True False False False]
 [ True False False  True  True]]
[[False  True False False False  True]
 [False False  True  True False False]
 [ True False  True  True  True  True]
 [False  True False  True  True False]
 [False False False  True False  True]]
[[-1  0  1 -1  0]
 [-1 -1 -1  1 -1]
 [ 0  0  1 -1 -1]
 [-1  0  1  0 -1]
 [-1  1  0  0  0]]


In [5]:
# DnB 환경의 상태, 행동을 DnB Env의 형태로 변환

def interpret_edges(edges) -> list[tuple[int,int]]:
    def map_func(e):
        if e == None:
            return 0
        else:
            return 1

    i_edges = [[map_func(e) for e in r] for r in edges]

    return i_edges

def interpret_box_owner(box_owner) -> list:
    def map_func(box):
        if box == 0:
            return -1
        elif box == None:
            return 0
        else :
            return box 
        
    for r in range(len(box_owner)):
        for c, e in enumerate(box_owner[r]):
            if e == 0:
                box_owner[r][c] = -1
    i_box_owner = [[map_func(box) for box in r] for r in box_owner]
    return i_box_owner


def interpret_action(action) -> tuple[str, int, int]:
    ori = 'H' if action[0] == 0 else 'V'
    r, c = map(int, action[1:])
    return (ori, r, c)

#### Test Code

In [6]:
DnB = DotsAndBoxes(5)
DnB.claim_edge(('V', 1, 1))
DnB.claim_edge(('H', 1, 1))
DnB.claim_edge(('H', 2, 3))

print("DnB_h_Edges\n", DnB.h_edges)
print("Interpreted h_Edges\n", interpret_edges(DnB.h_edges))

# Test For Deep Copy issue
print("DnB_v_Edges\n", DnB.h_edges)
print("Interpreted v_Edges\n", interpret_edges(DnB.h_edges))

1
0
DnB_h_Edges
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, 0, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted h_Edges
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
DnB_v_Edges
 [[None, None, None, None, None], [None, 1, None, None, None], [None, None, None, 0, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None]]
Interpreted v_Edges
 [[0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [7]:
DnB = DotsAndBoxes(5)
action = action_space.sample()
i_action = interpret_action(action)
print(i_action)
DnB.claim_edge(i_action)
print("DnB_h_Edges\n", DnB.h_edges)


action = action_space.sample()
i_action = interpret_action(action)
print(i_action)
DnB.claim_edge(i_action)
print("DnB_h_Edges\n", DnB.h_edges)

('H', 5, 0)
0
DnB_h_Edges
 [[None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [0, None, None, None, None]]
('V', 2, 4)
DnB_h_Edges
 [[None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [None, None, None, None, None], [0, None, None, None, None]]


In [ ]:
DnB = DotsAndBoxes(5)

DnB.claim_edge(('V', 1, 1))
DnB.claim_edge(('H', 1, 1))
DnB.claim_edge(('H', 2, 1))
DnB.claim_edge(('V', 1, 2))

print("DnB_box_owner\n", DnB.box_owner)
print("Interpreted box_owner\n", interpret_box_owner(DnB.box_owner))

# Test For Deep Copy issue
print("DnB_box_owner\n", DnB.box_owner)
print("Interpreted box_owner\n", interpret_box_owner(DnB.box_owner))

1
0


In [9]:
action = action_space.sample()
print(action)
print(interpret_action(action))

[1 1 1]
('V', 1, 1)


### DnBEnv

In [9]:
class DnBEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, render_mode=None, n_box = 5):
        self.n_box = n_box # grid size
        self.DnB = None # Game 인스턴스, 초기화는 reset에서 수행
        self.window_size = 512 #The size of the Pygame window

        self.observation_space = spaces.Dict(
            {
                "h_edges": spaces.Box(0, 1, shape=(6,5), dtype=bool),
                "v_edges": spaces.Box(0, 1, shape=(5,6), dtype=bool),
                "box_owner": spaces.Box(-1, 1, shape=(5,5), dtype=int)
            }
        )

        self.action_space = spaces.MultiDiscrete(nvec=[2,6,6], dtype=int)
        self.action_mask = None

        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode

        """
        If human-rendering is used, `self.window` will be a reference
        to the window that we draw to. `self.clock` will be a clock that is used
        to ensure that the environment is rendered at the correct framerate in
        human-mode. They will remain `None` until human-mode is used for the
        first time.
        """
        
        self.screen = None
        self.clock = None
        self.fonts = None


    def _get_obs(self) -> dict:

        obs = {
            'h_edges': interpret_edges(self.DnB.h_edges),
            'v_edges': interpret_edges(self.DnB.v_edges),
            'box_owner': interpret_box_owner(self.DnB.box_owner)
        }

        return obs

    # auxilary information.
    # action mask를 항상 포함한다.
    def _get_info(self) -> dict:
        
        return {
            'action_mask' : self.action_mask
        }
    
    def reset(self, seed=None, options=None) -> tuple[dict, dict]:
        super().reset(seed=seed)

        self.DnB = DotsAndBoxes(self.n_box)

        # 불가능한 테두리의 행동들을 마스킹한다.
        def get_init_action_mask(n_box) -> np.ndarray:
            # shape: (n_box+1, n_box+1)
            r = np.arange(n_box + 1)[:, None]        # (R,1)
            c = np.arange(n_box + 1)[None, :]        # (1,C)

            mask = np.zeros((2, n_box + 1, n_box + 1), dtype=bool)
            # Horizontal mask: True only when c == n_box
            
            mask[0] = (c == n_box)
            # Vertical mask: True only when r == n_box
            mask[1] = (r == n_box)
            # Stack so that mask[0] = H, mask[1] = V
            
            return mask
        self.action_mask = get_init_action_mask(self.n_box)


        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info
    
    # render_mode가 rgb_array인 경우에만 np.ndarray 반환
    def render(self) -> np.ndarray | None:
        if self.render_mode == "rgb_array":
            return self._render_frame()

    # frame을 render한다.
    # render_mode가 'human'인 경우에는 pygame 창에 그린다.
    def _render_frame(self) -> np.ndarray | None:
        if self.screen is None and self.clock is None and self.render_mode == 'human':
            pygame.init()
            pygame.display.init()
            self.screen, self.clock, self.fonts = get_render_desc(self.n_box)

        if self.render_mode == "human":
            draw_board(self.screen, self.DnB, self.fonts)
            pygame.event.pump()
            pygame.display.update()
            
            self.clock.tick(self.metadata['render_fps'])

        else:
            return np.transpose(
                np.array(pygame.surfarray.pixels3d(self.screen)), axes=(1, 0, 2)
            )
    
    # 행동 수행
    def step(self, action) -> tuple[dict, int, bool, bool, dict]:
        assert not self.action_mask[action[0], action[1], action[2]], "The action has already been taken. Choose another action."

        i_action = interpret_action(action)
        self.action_mask[action[0], action[1], action[2]] = True
        self.DnB.claim_edge(i_action)
        
        terminated = self.DnB.is_game_over
        reward = 1 if terminated else 0
        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, False, info
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [ ]:
n_box = 5
env = DnBEnv(render_mode='human', n_box=n_box)

observation, info = env.reset()
action_mask = info['action_mask']

print(f"Starting observation: {observation}")

episode_over = False
total_reward = 0

while not episode_over:

    action = env.action_space.sample()
    while action_mask[action[0], action[1], action[2]]:
        action = env.action_space.sample()
    
    print('action:', action)
    print(info['action_mask'])
    print('Number of Claimed Edges:', np.sum(action_mask == False))

    observation, reward, terminated, truncated, info = env.step(action)
    action_mask = info['action_mask']
            
    total_reward += reward
    episode_over = terminated or truncated

print(f"Episode finished! Winner: {info['winner']} Total reward: {total_reward}")
print(f'observation: {observation}')
print(f'info: {info}')


env.close()

Starting observation: {'h_edges': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'v_edges': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'box_owner': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]}
action: [0 1 1]
[[[False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]
  [False False False False False  True]]

 [[False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [False False False False False False]
  [ True  True  True  True  True  True]]]
Number of Claimed Edges: 60
0
action: [1 2 1]
[[[False False False False False  True]
  [False  True False False False  True]
  [False False False Fa